# CSA epoch15 → SCST_CE_OBJECT, two GPUs
        Run **SAVE_CSA_XE15_V147_DATASET** on CPU first. Then select **T4x2**, Internet ON,
        Save Version → Save & Run All. One arm per account; both notebooks use2GPUs.
        Attach4inputs: csa-full-roi-karpathy-v134; csa-full-xe15-v147;
        mscoco-2014 (dataset_coco.json only); objectdetectionecache. Edit mount paths in cell1.
        No API secrets, raw images, bank130 or new feature extraction in these GPU notebooks.

        CE +0.05SCST(CIDEr-D +0.5Robj).
        Same CSA15 weights, freshAdam5e-6, FP32, dropoutOFF, global8=4/GPU.
        Fixed500updates=4000unique TRAIN images drawn from a locked10k pool. This is a pilot,
        not a full RL epoch. FullTRAIN reference IDF; held-out labels excluded from reward.
        Original CSA alignment retained; no extra word-region attention loss.

        Final VALID5000 beam5/max20 with CIDEr, BLEU4, ROUGE-L, CHAIRs/i, recall, NLL.
        No TEST or mid-checkpoint selection. Four-hour stop cap is not an ETA/billing promise.
        Atomic checkpoints every100updates; measured throughput gate at25updates preserves
        partial weights if remaining budget is insufficient. Do not compare partial arms.
        Local author checks are NumPy/AST only; real policy/DDP/CUDA tests run here before training.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
ARM='SCST_CE_OBJECT'
WALL_HOURS=4 # Stop cap, NOT an ETA or billing estimate. User reports unchanged20h/23h.
FULL_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-roi-karpathy-v134')
CHECKPOINT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-xe15-v147')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
RESUME_ROOT='' # Same-arm prior output ONLY. Empty for first run; never automatically retries.
OUTPUT=Path('/kaggle/working/csa15_'+ARM.lower()+'_pilot500')
for p in (FULL_ROOT,CHECKPOINT_ROOT,COCO_JSON,DETECTIONS):
    if not p.exists():raise FileNotFoundError('Attach all4inputs before GPU: '+str(p))
probe="import json,torch;print(json.dumps(dict(version=torch.__version__,built=torch.backends.cuda.is_built(),available=torch.cuda.is_available(),count=torch.cuda.device_count())))"
GPU_ENV=json.loads(subprocess.check_output([sys.executable,'-c',probe],text=True,timeout=60))
print('GPU environment',GPU_ENV,flush=True)
if not GPU_ENV['built'] or not GPU_ENV['available'] or GPU_ENV['count']!=2:raise RuntimeError('Select GPU T4x2 and restart session; do not reinstall Torch blindly.')
if OUTPUT.exists():raise RuntimeError('Preserve old output; use a fresh output path/session.')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<8*1024**3:raise RuntimeError('Need8GiB free for stage/checkpoints.')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='csa15-scst-'));STAGE=SOURCE/'stage';LOG=OUTPUT/'notebook.log'
COMMIT='ff9a3b64705fdafb10897762b3013b36c337cee0'
HASHES={'experiments/csa_paper/scst_pilot_data.py': '52db5737ee947b06d8d8138c0c393bdd7d2f36c63657a69ad7c2260df818fc6b', 'experiments/csa_paper/scst_pilot_model.py': '0b5e2d384ffc7195c82888f3f7904eac145ba7291ebed27dc8342ed8e1ac9576', 'experiments/csa_paper/run_scst_pilot.py': '3089c2ae6ad291683830b27cbf70cc37f72976bb5d20196b9e8f94cd7619e2f9', 'experiments/csa_paper/test_scst_pilot_data.py': '7bd5a25fd260593c1cf9e435055c30f6beac906d885960c8955a018969d67b1b', 'experiments/csa_paper/test_scst_pilot_model.py': 'd1192a844ad111f3423d39f6cdece1daca88417b3255bd44fd1f096bc9588ed5', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/full_cache_data.py': '58cf4d08310fe03e01441af2ddeb12fd3404f59e82259194a32fe98c8b002bc1', 'experiments/csa_paper/full_xe_data.py': 'a7a5a3b4d70c9d3a357aa464dfe397a6cba8d851e33065299fb98721ddca7135', 'experiments/csa_paper/sharing_data.py': '68a065f5c055a55b41eae0ea554fbb7f8fd1124e8f5422679fcb1d4878f14829', 'experiments/csa_paper/package_full_roi_dataset.py': 'be3e7d0f96add450a5295add6466e19977da90f486e5e726e3ea40a505cceecb', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/SAVE_FULL_ROI_V134_PACKAGE.json': 'a73636ca75806d6b1d7b4d88f80f3ce1c59b3061eb3e9a5e730097ddd89c505c', 'experiments/csa_paper/SCST_PILOT_PROTOCOL.md': 'e53a8ee9792994cc329b65c0ea87b626f9d87e234af873884366b6c5a83ac0e8', 'experiments/csa_paper/make_scst_pilot_notebooks.py': '93cbacbe13de321e32010c80995933184a0ed4b4cd0ac26e999ceb271ca019b9', 'experiments/csa_paper/make_smoke_notebook.py': '9de13e286efb3ddd29a2eed331219a9e60dcb384fc2f28e062f64dd8dd1c6054', 'captioning/__init__.py': 'ad259f50a32265d298d027a888f801d378c4669313b8dab06e0653978ce30633', 'captioning/scst.py': '3a9a1f32fb27dae0447066e1388a31461f3f0781d52db5861880b037a572a78e', 'captioning/object_reward.py': '23282b1faca0b8e177ad8ce300f0b78ad4a5ab7b100e4b359b044025c567800c', 'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73'}
MAPPING={'experiments/csa_paper/scst_pilot_data.py': 'scst_pilot_data.py', 'experiments/csa_paper/scst_pilot_model.py': 'scst_pilot_model.py', 'experiments/csa_paper/run_scst_pilot.py': 'run_scst_pilot.py', 'experiments/csa_paper/test_scst_pilot_data.py': 'test_scst_pilot_data.py', 'experiments/csa_paper/test_scst_pilot_model.py': 'test_scst_pilot_model.py', 'experiments/csa_paper/model.py': 'model.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/full_cache_data.py': 'full_cache_data.py', 'experiments/csa_paper/full_xe_data.py': 'full_xe_data.py', 'experiments/csa_paper/sharing_data.py': 'sharing_data.py', 'experiments/csa_paper/package_full_roi_dataset.py': 'package_full_roi_dataset.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/SAVE_FULL_ROI_V134_PACKAGE.json': 'SAVE_FULL_ROI_V134_PACKAGE.json', 'experiments/csa_paper/SCST_PILOT_PROTOCOL.md': 'SCST_PILOT_PROTOCOL.md', 'experiments/csa_paper/make_scst_pilot_notebooks.py': 'make_scst_pilot_notebooks.py', 'experiments/csa_paper/make_smoke_notebook.py': 'make_smoke_notebook.py', 'captioning/__init__.py': 'captioning/__init__.py', 'captioning/scst.py': 'captioning/scst.py', 'captioning/object_reward.py': 'captioning/object_reward.py', 'captioning/object_consistency.py': 'captioning/object_consistency.py'}
for name,expected in HASHES.items():
    raw=urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/{name}',timeout=90).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned source:'+name)
    path=SOURCE/MAPPING[name];path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)
os.environ['OMP_NUM_THREADS']='1'
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT,sha256=HASHES),indent=2),encoding='utf-8')
(OUTPUT/'gpu_environment.json').write_text(json.dumps(GPU_ENV),encoding='utf-8')
CHAIR=SOURCE/'chair';CONFIG=SOURCE/'config.json'
cfg=dict(arm=ARM,full_root=str(FULL_ROOT),checkpoint_root=str(CHECKPOINT_ROOT),dataset_json=str(COCO_JSON),detections=str(DETECTIONS),
    output=str(OUTPUT),stage=str(STAGE),chair=str(CHAIR),resume_root=RESUME_ROOT,worker_seconds=9000)
CONFIG.write_text(json.dumps(cfg),encoding='utf-8')


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('CSA SCST active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Smoke wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False;deadline=time.monotonic()+WALL_HOURS*3600
try:
    run([sys.executable,'-u',SOURCE/'test_scst_pilot_data.py'],timeout=120)
    run([sys.executable,'-m','pip','install','-q','pycocoevalcap==1.2','nltk==3.9.2','future'],timeout=300)
    run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],timeout=300)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],timeout=300)
    run(['java','-version'],timeout=30)
    run(['git','init',CHAIR],timeout=60)
    run(['git','-C',CHAIR,'remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],timeout=60)
    run(['git','-C',CHAIR,'fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],timeout=300)
    run(['git','-C',CHAIR,'checkout','--detach','FETCH_HEAD'],timeout=60)
    run([sys.executable,'-u',SOURCE/'test_scst_pilot_model.py'],timeout=180)
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    run([*launch,SOURCE/'test_scst_pilot_model.py','--ddp'],timeout=300)
    run([sys.executable,'-u',SOURCE/'run_scst_pilot.py','prepare','--config',CONFIG],timeout=min(2400,deadline-time.monotonic()-1800))
    run([sys.executable,'-u',SOURCE/'run_scst_pilot.py','metric_preflight','--config',CONFIG],timeout=300)
    remaining=deadline-time.monotonic()
    if remaining<2400:raise TimeoutError('Not enough worker/scoring margin; do not train automatically.')
    cfg['worker_seconds']=remaining-900;CONFIG.write_text(json.dumps(cfg),encoding='utf-8')
    run([*launch,SOURCE/'run_scst_pilot.py','worker','--config',CONFIG],timeout=cfg['worker_seconds']+60)
    run([sys.executable,'-u',SOURCE/'run_scst_pilot.py','score','--config',CONFIG],timeout=min(900,deadline-time.monotonic()))
    complete=json.loads((OUTPUT/'result.json').read_text(encoding='utf-8'))['status']=='CSA_SCST_PILOT_COMPLETE'
    if not complete:raise RuntimeError('Incomplete endpoint/VALID')
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,arm=ARM,updates_requested=500,test_used=False)),encoding='utf-8')
    archive=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file() and p.suffix in ('.json','.jsonl','.log','.txt') and p.name not in ('training_references.json','object_targets.json'):
                z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',archive,flush=True)
    print('KEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    print('KEEP CHECKPOINT:',OUTPUT/'checkpoint.bin' if (OUTPUT/'checkpoint.bin').is_file() else 'No checkpoint yet',flush=True)
    print('COMPLETE fixed500 + VALID5000' if complete else 'INCOMPLETE; preserve logs/weights; no automatic restart.',flush=True)
